In [1]:
## This script will ID and generate "warm season" and "cold season" masks for each weather station
### Warm season is defined by Hu, 2021 (doi: 10.1029/2021GL093260) as the 4-month period with the highest average T. Inverse for "cold season"
### 1/15/25

In [2]:
# imports
import os
import xarray as xr
import numpy as np
import netCDF4 
import glob
import pandas as pd
import geopandas as gpd
from datetime import datetime
from scipy import stats

/opt/sw/anaconda3/2023.09/envs/pangeo23/lib/python3.11/site-packages/pyproj/__init__.py:89: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()


In [3]:
# interactive plotting stuff 
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import colors
import matplotlib.cm as cm
import matplotlib.gridspec as gridspec
from matplotlib import rcParams
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

#import matplotlib.dates as mdates
%matplotlib inline
plt.rcParams['figure.figsize'] = 12, 6
#%config InlineBackend.figure_format = 'retina'

import cartopy
import cartopy.crs as ccrs
from cartopy.util import add_cyclic_point

In [4]:
def gen_seasonmask(idx, da):
    """
    This fn. generates a mask for a 4-month season which starts on month number [idx], 
    using dates provided in the 'time' dimension of the dataArray [dat]
    
    Args:
    -----
        idx: int
        da: xr.DataArray
        
    Returns:
    --------
        szn_mask: xr.DataArray (boolean)
        szn_yr: xr.DataArray (a new time dimension you can use)
        
    11/12/24
    """
    
    # create a mask for this station's "warm/cold season"
    if idx >= 10:
        szn_mask = (da.time.dt.month >= idx) + (da.time.dt.month <= (4 - ((12 - idx) + 1))) # month selection 'wraps around' dec-jan
        szn_yr = xr.where(da.time.dt.month>=idx, da.time.dt.year, (da.time.dt.year-1)) # 'season-year' var. ID's the year that the season started in (for grouping)
    else: 
        szn_mask = (da.time.dt.month >= idx) * (da.time.dt.month < idx+4) # month thru month+4
        szn_yr = da.time.dt.year
        
    return szn_mask, szn_yr

In [5]:
script = '/home/nsiegert/projects/coastal_sst/code/dataprep/warmszn_coldszn_masks.ipynb'

In [6]:
# dataframe with the stations we are using
df = pd.read_csv('/home/nsiegert/projects/coastal_sst/data/hadisd_stations_using_Expanded.csv')
df = df.drop(['Unnamed: 0'], axis=1)
df

,STAID,STANAME,LAT,LON,ELEV,START,END,DIST2COAST,PCTREPORTING,max_pctmissing_month
0,010014-99999,SORSTOKKEN,59.792,5.341,48.8,1986-11-20,2024-02-02,1.377415,82.28,26.021505
1,011020-99999,SKLINNA FYR,65.200,11.000,16.0,1975-02-28,2024-02-02,23.468849,92.73,16.646989
2,011120-99999,BRONNOY,65.461,12.218,7.6,1973-01-01,2024-02-02,0.976040,96.37,12.514758
3,011160-99999,STOKKA,65.950,12.467,17.1,1973-04-01,2024-02-02,1.240810,95.79,15.702479
4,011210-99999,SOLVAER III,66.367,12.617,10.0,1973-01-01,2024-02-02,10.035263,87.32,17.222222
...,...,...,...,...,...,...,...,...,...,...
1469,987520-99999,BUTUAN,8.950,125.483,46.0,1981-01-01,2024-02-02,4.531335,96.62,6.666667
1470,987530-99999,FRANCISCO BANGOY INTL,7.126,125.646,29.3,1974-06-01,2024-02-02,1.183564,99.53,3.111111
1471,987550-99999,HINATUAN,8.367,126.333,3.0,1949-10-01,2023-12-07,1.553108,96.70,6.021505
1472,988360-99999,ZAMBOANGA INTL,6.922,122.060,10.1,1945-03-12,2024-02-02,2.489244,99.64,2.333333


In [7]:
## make climatology of tmean for each station ##

# load tmin & tmax
tx_da = xr.open_dataset('/dx02/data/nsiegert/coastal_mhw_data/ALLSTATIONS.tx.nc').Tx
tn_da = xr.open_dataset('/dx02/data/nsiegert/coastal_mhw_data/ALLSTATIONS.tn.nc').Tn

# compute tmean
tmean_da = ((tx_da + tn_da) / 2)

# tmean climatology
tmeanclim = tmean_da.groupby("time.month").mean(dim='time')
tmeanclim

<xarray.DataArray (staid: 1474, month: 12)> Size: 142kB
array([[ 2.86786114,  2.66251557,  4.14763552, ...,  9.28938731,
         5.72834646,  3.35838509],
       [ 2.05116959,  1.61804878,  2.32796235, ...,  7.6003692 ,
         4.57760801,  2.74784946],
       [ 0.82764188,  0.45940594,  1.71243806, ...,  7.06000981,
         3.65924242,  1.74169154],
       ...,
       [26.50802469, 26.59350105, 26.98633301, ..., 28.16858142,
        27.5216    , 27.08596404],
       [27.9161597 , 28.12703549, 28.43828273, ..., 28.16792183,
        28.33588235, 28.19288425],
       [15.50028517, 17.03192708, 20.11816888, ..., 25.01288023,
        20.30394089, 16.70754269]])
Coordinates:
  * staid    (staid) <U12 71kB '010014-99999' '011020-99999' ... '999999-12946'
  * month    (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12

In [8]:
# get the tmean climatology extended so that we can consider warm seasons starting in oct/nov/december
repeat2 = np.zeros(shape=(len(tmean_da.staid), 15))
repeat2[:, 0:12] = tmeanclim.data
repeat2[:, 12:15] = tmeanclim.data[:, :3]

In [9]:
# appy 4 month rolling window to compute 4-month running tmean
repeat2da = xr.DataArray(repeat2, dims=['staid', 'mo'], coords={'staid': tmean_da.staid, 'mo': np.array(range(1, 16))})
repeat2da_roll = repeat2da.sortby('mo', ascending=False).rolling(mo=4).mean() 
# ^ (I have the month dim sorted inversely here b/c I want the rolling window to be indexed to the start rather than end of window)
# (e.g. data for mo=1 represents rolling JFMA data)

In [10]:
# get indices to the warm and cold seasons for each staid
warm_szn_idxs = repeat2da_roll.idxmax(skipna=True, dim='mo').astype(int)
cold_szn_idxs = repeat2da_roll.idxmin(skipna=True, dim='mo').astype(int)

warm_szn_idxs.name='warm_idx'
cold_szn_idxs.name='cold_idx'

In [11]:
# generate warm season and cold season masks for each station
warm_mask_arr = np.zeros(shape=tmean_da.shape)
cold_mask_arr = np.zeros(shape=tmean_da.shape)

for i in range(len(tmean_da.staid)):
    
    warm_idx = warm_szn_idxs[i].item()
    cold_idx = cold_szn_idxs[i].item()
    
    warm_mask, warmszn_years = gen_seasonmask(warm_idx, tmean_da)
    cold_mask, coldszn_years = gen_seasonmask(cold_idx, tmean_da)
    
    warm_mask_arr[i] = warm_mask.data
    cold_mask_arr[i] = cold_mask.data
    
    if i%100==0: print(i)

0
100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400


In [12]:
# put into DataArrays
warm_mask_da = xr.DataArray(warm_mask_arr.astype(int), dims=tmean_da.dims, coords=tmean_da.coords, name='warmszn_mask')
cold_mask_da = xr.DataArray(cold_mask_arr.astype(int), dims=tmean_da.dims, coords=tmean_da.coords, name='coldszn_mask')

In [13]:
# put all into one dataset
ds_out = xr.merge([warm_mask_da, cold_mask_da, warm_szn_idxs, cold_szn_idxs])

In [14]:
# add attrs
ds_out.attrs['desc.'] = 'Warm season is defined by Hu, 2021 (doi: 10.1029/2021GL093260) as the 4-month period with the highest average T. Inverse for "cold season"'
ds_out.attrs['script'] = script
now = datetime.now()
ds_out.attrs['timestamp'] = now.strftime("%Y-%m-%d %H:%M:%S")

In [15]:
# save
ds_out.to_netcdf('/dx02/data/nsiegert/coastal_mhw_data/ALLSTATIONS.warm_cold_seasons.nc')

In [16]:
print('done')

done


---

In [ ]:
# old fn. that was helpful but I rewrote to be faster. 

In [17]:
# function defs

def find_warm_season(sta):
    """
    This function finds the warm season for a given weather station. 
    Warm season is defined by Hu, 2021 (doi: 10.1029/2021GL093260) as the 4-month period with the highest average T.
    
    Params:
    -------
        sta (string or xarray dataset) can supply the HadISD station ID (ie '722020-12839') or xarray dataset for a weather station.
        
    Returns:
    --------
        warmest_idx (int) month number to the start of the warm season. Ex. if warm season is JJAS, this will return 6. 
        
    I've checked this fn. to the Hu supmat for various shared weather stations and the results agree.
    6/26/24
    """

    if type(sta) == str:

        # open dataset for that station
        sta_ds = xr.open_dataset('/dx01/nsiegert/HadISD/daily/{}_daily.nc'.format(staid))

    elif str(type(sta)) == "<class 'xarray.core.dataset.Dataset'>":
        
        sta_ds = sta
    
    # open dataset for that station
    sta_ds = xr.open_dataset('/dx01/nsiegert/HadISD/daily/{}_daily.nc'.format(staid))
    
    # make tmean climatology
    tmean = ((sta_ds.tmax + sta_ds.tmin) / 2)
    tmeanclim = tmean.groupby("time.month").mean(dim='time')
    
    # get the tmean climatology extended so that we can consider warm seasons starting in oct/nov/december
    repeat2 = np.zeros(15)
    repeat2[0:12] = tmeanclim.data
    repeat2[12:15] = tmeanclim.data[:3]
    
    
    # loop through and find the index to the season 
    seasonmean = 0
    warmest_idx = 0

    for s_idx in range(12):

        # compute season mean T
        mn = np.mean(repeat2[s_idx:(s_idx+4)])

        # if warmer, update the warmest season inded
        if mn > seasonmean:
            seasonmean = mn
            warmest_idx = s_idx
    
    # return index to the start of warmest 4 months (+1, b/c index starts at 0, months start at 1)
    return (warmest_idx + 1)


def find_cold_season(sta):
    """
    This function finds the COLD season for a given weather station. 
    Warm season is defined opposite Hu, 2021 (doi: 10.1029/2021GL093260) as the 4-month period with the lowest average T.
    
    Params:
    -------
        sta (string or xarray dataset) can supply the HadISD station ID (ie '722020-12839') or xarray dataset for a weather station.
        
    Returns:
    --------
        coldest_idx (int) month number to the start of the warm season. Ex. if season is JJAS, this will return 6. 
        
    11/12/24
    """

    if type(sta) == str:

        # open dataset for that station
        sta_ds = xr.open_dataset('/dx01/nsiegert/HadISD/daily/{}_daily.nc'.format(staid))

    elif str(type(sta)) == "<class 'xarray.core.dataset.Dataset'>":
        
        sta_ds = sta
    
    # open dataset for that station
    sta_ds = xr.open_dataset('/dx01/nsiegert/HadISD/daily/{}_daily.nc'.format(staid))
    
    # make tmean climatology
    tmean = ((sta_ds.tmax + sta_ds.tmin) / 2)
    tmeanclim = tmean.groupby("time.month").mean(dim='time')
    
    # get the tmean climatology extended so that we can consider warm seasons starting in oct/nov/december
    repeat2 = np.zeros(15)
    repeat2[0:12] = tmeanclim.data
    repeat2[12:15] = tmeanclim.data[:3]
    
    
    # loop through and find the index to the season 
    seasonmean = 11111111
    coldest_idx = 0

    for s_idx in range(12):

        # compute season mean T
        mn = np.mean(repeat2[s_idx:(s_idx+4)])

        # if colder, update the warmest season inded
        # (why didn't I just apply a 4month rolling window and then use argmin? who knows - 11/12/24)
        if mn < seasonmean:
            seasonmean = mn
            coldest_idx = s_idx
    
    # return index to the start of coldest 4 months (+1, b/c index starts at 0, months start at 1)
    return (coldest_idx + 1)